In [0]:
from pyspark.sql.functions import *

data = [
(1," John ","M","5000","2024-01-01","India"),
(2,"Mary","F","7000","2024-01-02","India"),
(3,"Sam",None,"-200","2024-01-03","USA"),
(4,None,"M","4000","2024-01-04","UK"),
(1," John ","M","5000","2024-01-01","India"),   # duplicate
(5,"Alice","f","abc","2024-01-05","India")      # invalid salary
]

columns = ["customer_id","name","gender","salary","join_date","country"]

bronze_df = spark.createDataFrame(data,columns)

bronze_df.show()

In [0]:
bronze_df.printSchema()

In [0]:
df = bronze_df.withColumn(
    "salary", col("salary").cast("double")
).withColumn(
    "join_date", to_date("join_date", "yyyy-MM-dd")
)

In [0]:
df.printSchema()

In [0]:
df = df.withColumn(
    "name", trim(col("name"))
)

In [0]:
df.display()

In [0]:
df = bronze_df.withColumn(
    "salary",
    expr("try_cast(salary as double)")
)

In [0]:
df.display()

In [0]:
df = df.withColumn(
    "name", trim(col("name"))
)

In [0]:
df.display()

In [0]:
df = df.withColumn(
    "gender",
    when(lower(col("gender")) == "m", "Male")
    .when(lower(col("gender")) == "f", "Female")
)

In [0]:
df.display()

In [0]:
df =df.dropDuplicates(["customer_id", "join_date"])

In [0]:
df.display()

In [0]:
df.filter(
    col("customer_id").isNull() |
    col("name").isNull()
).show()

In [0]:
valid_df = df.filter(col("name").isNotNull())

In [0]:
valid_df.display()

In [0]:
valid_df = valid_df.filter(col("salary") > 0)

In [0]:
from pyspark.sql.functions import *


data = [
(1," John ","5000","2024-01-01","India"),
(2,"Mary","7000","2024-02-01","India"),
(3,"Sam","abc","2024-02-10","USA"),      # bad salary
(4,"Alice","4500","2024-99-01","UK"),    # bad date
(5,None,"3000","2024-03-01","India"),    # null name
(1," John ","5000","2024-01-01","India") # duplicate
]

columns = ["customer_id","name","salary","join_date","country"]

bronze_df = spark.createDataFrame(data,columns)

bronze_df.show()

In [0]:
df = bronze_df.withColumn(
    "name", 
    trim(col("name"))
)

In [0]:
df = df.dropDuplicates(["customer_id","join_date"])

In [0]:
df.display()

In [0]:
df = df.withColumn(
"salary_double",
expr("try_cast(salary as double)")
).withColumn(
"join_date_clean",
expr("try_cast(join_date as date)")
)

In [0]:
df.display()

In [0]:
#Create a validation flag

df = df.withColumn(
    "is_valid",
    (col("customer_id").isNotNull()) &
    (col("name").isNotNull()) &
    (col("salary_double").isNotNull()) &
    (col("join_date_clean").isNotNull())
)

In [0]:
df.display()